# keyboard_config

> Keyboard navigation configuration for the Phase 2 decomposition step

In [ ]:
#| default_exp components.step_decomposition.keyboard_config

In [ ]:
#| export
from typing import Any

from fasthtml.common import Div, Details, Summary

# DaisyUI components
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui
from cjm_fasthtml_daisyui.components.data_display.collapse import (
    collapse, collapse_title, collapse_content, collapse_modifiers
)

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.core.base import combine_classes

# Keyboard navigation library
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.hints import render_keyboard_hints

# Card stack library
from cjm_fasthtml_card_stack.core.config import CardStackConfig
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds
from cjm_fasthtml_card_stack.core.button_ids import CardStackButtonIds
from cjm_fasthtml_card_stack.keyboard.actions import (
    create_card_stack_focus_zone, create_card_stack_nav_actions,
)

# Token selector library
from cjm_fasthtml_token_selector.keyboard.actions import (
    create_token_selector_mode, create_token_nav_actions,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_TS_CONFIG,
)

## Hidden Input and Button IDs

Constants for hidden inputs and HTMX trigger buttons used by the keyboard navigation system.

In [ ]:
#| export
# Button IDs for keyboard actions — decomposition operations (workflow-specific)
SD_DECOMP_ENTER_SPLIT_BTN = "sd-decomp-enter-split-btn"
SD_DECOMP_EXIT_SPLIT_BTN = "sd-decomp-exit-split-btn"
SD_DECOMP_SPLIT_BTN = "sd-decomp-split-btn"
SD_DECOMP_MERGE_BTN = "sd-decomp-merge-btn"
SD_DECOMP_UNDO_BTN = "sd-decomp-undo-btn"

## Decomposition Keyboard Configuration (Consumer-Specific)

Factory function to create the full keyboard manager for Phase 2, composing
generic card stack navigation with decomposition-specific split mode and operations.

In [ ]:
#| export
def _create_decomposition_keyboard_manager(
    ids:CardStackHtmlIds,  # Card stack HTML IDs
    button_ids:CardStackButtonIds,  # Card stack button IDs for navigation
    config:CardStackConfig,  # Card stack configuration
) -> ZoneManager:  # Configured keyboard zone manager
    """Create the keyboard zone manager for Phase 2 decomposition step.

    Composes library-provided card stack navigation actions with decomposition-specific
    split mode, merge, undo, and token selector navigation actions.
    """
    # Card stack zone from library
    card_zone = create_card_stack_focus_zone(
        ids=ids,
        on_focus_change="onCardFocusChange",
        data_attributes=("segment-index",),
    )
    zone_id = card_zone.id

    # Card stack navigation actions from library (arrows, page jump, first/last, width)
    nav_actions = create_card_stack_nav_actions(
        zone_id=zone_id,
        button_ids=button_ids,
        config=config,
        disable_in_modes=("split",),
    )

    # --- Token selector: split mode definition (library factory) ---
    split_mode = create_token_selector_mode(
        DECOMP_TS_CONFIG,
        mode_name="split",
        indicator_text="Split Mode",
        exit_key="",
        exit_on_zone_change=False,
    )

    # --- Token selector: Home/End navigation in split mode ---
    token_nav_actions = create_token_nav_actions(
        DECOMP_TS_CONFIG,
        zone_id=zone_id,
        mode_name="split",
    )

    # --- Consumer-specific: decomposition operation actions ---
    decomp_actions = (
        # Enter split mode (Enter or Space when NOT in split mode)
        KeyAction(
            key="Enter",
            htmx_trigger=SD_DECOMP_ENTER_SPLIT_BTN,
            mode_enter="split",
            not_modes=("split",),
            description="Enter split mode",
            hint_group="Decomposition",
        ),
        KeyAction(
            key=" ",
            htmx_trigger=SD_DECOMP_ENTER_SPLIT_BTN,
            mode_enter="split",
            not_modes=("split",),
            description="Enter split mode",
            hint_group="Decomposition",
            show_in_hints=False,
        ),

        # Execute split (Enter or Space when IN split mode)
        KeyAction(
            key="Enter",
            htmx_trigger=SD_DECOMP_SPLIT_BTN,
            mode_exit=True,
            mode_names=("split",),
            description="Split at caret",
            hint_group="Split Mode",
        ),
        KeyAction(
            key=" ",
            htmx_trigger=SD_DECOMP_SPLIT_BTN,
            mode_exit=True,
            mode_names=("split",),
            description="Split at caret",
            hint_group="Split Mode",
            show_in_hints=False,
        ),

        # Exit split mode (Escape when IN split mode)
        KeyAction(
            key="Escape",
            htmx_trigger=SD_DECOMP_EXIT_SPLIT_BTN,
            mode_exit=True,
            mode_names=("split",),
            description="Exit split mode",
            hint_group="Split Mode",
        ),

        # Merge with previous (Backspace when NOT in split mode)
        KeyAction(
            key="Backspace",
            htmx_trigger=SD_DECOMP_MERGE_BTN,
            not_modes=("split",),
            description="Merge with previous",
            hint_group="Decomposition",
        ),

        # Undo (Ctrl+Z in any mode)
        KeyAction(
            key="z",
            modifiers=frozenset({"ctrl"}),
            htmx_trigger=SD_DECOMP_UNDO_BTN,
            description="Undo",
            hint_group="General",
        ),
    )

    return ZoneManager(
        zones=(card_zone,),
        actions=(*nav_actions, *decomp_actions, *token_nav_actions),
        modes=(split_mode,),
        prev_zone_key="",
        next_zone_key="",
        state_hidden_inputs=True,
    )

## Keyboard Hints Component

In [ ]:
#| export
def _render_decomposition_keyboard_hints(
    manager: ZoneManager,  # Keyboard zone manager with actions configured
    container_id: str = "sd-decomp-keyboard-hints",  # HTML ID for the hints container
) -> Any:  # Collapsible keyboard hints component
    """Render keyboard shortcut hints in a collapsible container.
    
    Card-stack generic: renders hints for any ZoneManager in a DaisyUI collapse.
    The `container_id` parameter allows customization per workflow.
    """
    hints = render_keyboard_hints(
        manager,
        include_navigation=True,
        include_zone_switch=False,  # Single zone
        badge_style="outline",
        container_id=container_id,
        use_icons=False
    )
    
    return Details(
        Summary(
            "Keyboard Shortcuts",
            cls=combine_classes(collapse_title, font_size.sm, font_weight.medium)
        ),
        Div(
            hints,
            cls=collapse_content
        ),
        cls=combine_classes(collapse, collapse_modifiers.arrow, bg_dui.base_200)
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()